In [1]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from imblearn.under_sampling import RandomUnderSampler
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from sklearn.metrics import classification_report
from torch.autograd import detect_anomaly
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, \
    GenerationConfig
import google.generativeai as genai
from google.generativeai import types
import os
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import google.api_core.exceptions
from dotenv import load_dotenv
from openai import OpenAI
from datetime import datetime
from sklearn.linear_model import LogisticRegression
from accelerate import Accelerator

/home/cs/grad/islams32/dev/project/academic/technical-debt/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
accelerator = Accelerator()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
detect_train_df = pd.read_csv('../data/detect_train.csv')
detect_train_dataset = Dataset.from_pandas(detect_train_df)

detect_test_df = pd.read_csv('../data/detect_test.csv')
detect_test_dataset = Dataset.from_pandas(detect_test_df)

# detect_train_balanced_df = pd.read_csv('../data/detect_train_balanced.csv')
# detect_train_balanced_dataset = Dataset.from_pandas(detect_train_balanced_df)

detect_n_shot_df = pd.read_csv('../data/detect_n_shot.csv')
detect_n_shot_dataset = Dataset.from_pandas(detect_n_shot_df)

# Classification Dataset

In [4]:
classify_train_df = pd.read_csv('../data/classify_train.csv')
classify_train_dataset = Dataset.from_pandas(classify_train_df)

classify_test_df = pd.read_csv('../data/classify_test.csv')
classify_test_dataset = Dataset.from_pandas(classify_test_df)

classify_n_shot_df = pd.read_csv('../data/classify_n_shot.csv')
classify_n_shot_dataset = Dataset.from_pandas(classify_n_shot_df)

In [5]:
from util import get_first_n_line, get_last_n_line
from jinja2 import Template


class PromptTemplate:
    def __init__(self, name, definition, instruction, n_shot_template, line_m_before, line_n_after):
        self._name = name
        self._definition = definition
        self._instruction = instruction
        self._n_shot_template = n_shot_template
        self._line_m_before = line_m_before
        self._line_n_after = line_n_after

    @property
    def name(self):
        return self._name

    @property
    def definition(self):
        return self._definition

    @property
    def instruction(self):
        return self._instruction

    @property
    def line_m_before(self):
        return self._line_m_before

    @property
    def line_n_after(self):
        return self._line_n_after

    @property
    def shot_template(self):
        return self._n_shot_template

    def create_example(self, args):
        properties = dict(args)
        if 'code_before' in properties:
            properties['code_before'] = get_last_n_line(args['code_before'], self.line_m_before)
        if 'code_after' in properties:
            properties['code_after'] = get_first_n_line(args['code_after'], self.line_n_after)

        return Template(self.shot_template).render(**properties)

    def create_prompt(self, examples: [str]):
        return self.definition + "\n" + self.instruction + "\n" + "\n" + "\n\n".join(examples)

    def __repr__(self):
        return f"PromptTemplate(name={self.name}, description='{self.definition}', example='{self.shot_template}')"

In [6]:
from enum import Enum


class TrainStrategy(Enum):
    N_SHOT_RANDOM = 'n_shot_random'
    N_SHOT_SIMILAR = 'n_shot_similar'
    N_SHOT_TOP = 'n_shot_top'
    ALL = 'all'


In [7]:
import random
from sentence_transformers import SentenceTransformer, util

sentence_transformer = SentenceTransformer('all-MiniLM-L6-v2')


def pick_n_shot(train_dataset: Dataset, test_dataset: Dataset, test_index: int, n: int = 0,
                strategy: TrainStrategy = None):
    dataset_length = train_dataset.num_rows
    if dataset_length < n:
        raise Exception(f'Train dataset contains only {dataset_length} examples for {n} shots')
    indexes = []
    if strategy == TrainStrategy.N_SHOT_RANDOM:
        indexes = random.sample(dataset_length, n)
    elif strategy == TrainStrategy.N_SHOT_SIMILAR:
        similarities = util.cos_sim(sentence_transformer.encode(test_dataset['text'][test_index]),
                                    sentence_transformer.encode(train_dataset['text'])).squeeze(0).numpy()
        top_n_indices = np.argpartition(similarities, -n)[-n:]
        indexes = top_n_indices[np.argsort(similarities[top_n_indices])[::-1]].tolist()
    elif strategy == TrainStrategy.N_SHOT_TOP:
        indexes = [i for i in range(n)]
    return indexes


In [8]:
pick_n_shot(detect_train_dataset, detect_n_shot_dataset, 1, 1, TrainStrategy.N_SHOT_SIMILAR)
pick_n_shot(detect_train_dataset, detect_n_shot_dataset, 1, 1, TrainStrategy.N_SHOT_SIMILAR)

[22672]

In [9]:
def report_mismatch(file: str):
    _, name = os.path.basename(file).split('$', 1)
    merged_file = f'{os.path.dirname(file)}/merged_{name}'
    mismatched_file = f'{os.path.dirname(file)}/mismatched_{name}'
    last_df = pd.read_csv(file)

    for f in [merged_file, mismatched_file]:
        if not os.path.exists(merged_file):
            pd.DataFrame(columns=last_df.columns).to_csv(f, index=False)

    merged_df = pd.read_csv(merged_file)
    ids = merged_df['id'].values
    for index, row in last_df.iterrows():
        if row['id'] in ids:
            merged_df.loc[merged_df['id'] == row['id'], 'label_pred'] = row['label_pred']
        else:
            merged_df.loc[len(merged_df)] = row
    merged_df.sort_values(by=['id'], ascending=True, inplace=True)
    merged_df.to_csv(merged_file, index=False)
    merged_df[merged_df['label'] != merged_df['label_pred']].to_csv(mismatched_file, index=False)

In [10]:
def print_classification_excluding_outlier_repository(input_file: str, repository_id: int = 69):
    result_df = pd.read_csv(input_file)
    filtered_result_df = result_df[result_df['repository'] != repository_id]
    print(f'Test Result Excluding repository: {repository_id}')
    print(classification_report(filtered_result_df['label'], filtered_result_df['label_pred'], zero_division=0, digits=3))

In [11]:
from typing import List
from abc import abstractmethod

N_SHOT_PROPERTIES = ['text', 'label', 'code_before', 'code_after', 'cot']


class Model:
    def __init__(self, task_type: str, model_uri: str, known_labels: set[str], unmatched_label: str):
        self.task_type = task_type
        self.model_uri = model_uri
        self.known_labels = set([label.lower() for label in known_labels])
        self.unmatched_label = unmatched_label
        self.unknown_labels = []

    @abstractmethod
    def fit(self, dataset: Dataset):
        pass

    @abstractmethod
    def predict(self, dataset: Dataset):
        pass

    def format_label(self, label, verbose: bool = False):
        if label.lower() in self.known_labels:
            return label.lower()
        else:
            if verbose:
                print(f'Unknown Label: {label}')
            self.unknown_labels.append(label)
            return self.unmatched_label

    def predict_start(self, dataset: Dataset):
        print(f'{self.task_type} with {self.model_uri.split("/")[-1]}')
        self.unknown_labels.clear()

    def predict_end(self, dataset: Dataset, label_predictions):
        file_name = f'{self.task_type}_{self.model_uri.split("/")[-1]}'
        test_output = dataset.to_dict()
        test_output['label_pred'] = label_predictions
        if self.unknown_labels:
            print(f'Unknown Labels:\n{self.unknown_labels}')
        timestamp = datetime.now().strftime("%B %d, %Y, %H:%M:%S")
        file = f'{os.getenv("CACHE_DIRECTORY")}/{timestamp}${file_name}.csv'
        print(file)
        print('Test Result:')
        print(classification_report(dataset['label'], label_predictions, zero_division=0, digits=3))
        Dataset.from_dict(test_output).to_pandas().to_csv(file, index=False)
        report_mismatch(file)
        
        return file

    def project_properties(self, dataset: Dataset, index: int):
        properties = {}
        for key in N_SHOT_PROPERTIES:
            if key in dataset.features.keys():
                properties[key] = dataset[key][index]
        return properties

    def create_prompt(self, prompt_template: PromptTemplate, train_dataset: Dataset, train_indexes: [int],
                      test_dataset: Dataset,
                      test_index: int, verbose: bool = False):
        examples = [prompt_template.create_example(self.project_properties(train_dataset, index)) for index in
                    train_indexes]

        test_properties = self.project_properties(test_dataset, test_index)
        if 'label' in test_properties:
            test_properties['label'] = ''
        examples.append(prompt_template.create_example(test_properties))

        prompt = prompt_template.create_prompt(examples)
        if verbose:
            print(f'Prompt:\n {prompt}')
        return prompt

In [15]:
DETECTION_TEMPLATE = PromptTemplate(
    name="Manually Crafted",
    definition="You are Code Expert trained to detect Self-Admitted Technical Debt (SATD) in Java test code comments. SATD occurs when developers explicitly acknowledge that the current implementation is suboptimal, requires improvement, or contains technical compromises. These comments often include markers (e.g., TODO, FIXME), indicate unresolved issues, temporary fixes (e.g., workarounds, hacks), performance concerns, deprecated API usage, unsupported features, poor design, skipped tests, or unknown reasons. However, do not classify comments that merely describe expected behavior, actions, instructions or simply issue references, unless there is additional information indicating the need for future improvement. These comments often appear as imperative sentence structures (e.g., check argument, should not match), vague single-word description(e.g., clean up, retry, fail).",
    instruction="Classify by labelling it as 'yes' if the comment include a strong indication of Self-Admitted Technical Debt otherwise label it as 'no', do not return reason. Do not provide a reason for the classification.",
    n_shot_template="""
    <EXAMPLE>
    Comment: {{ text }}
    {% if cot -%}
    Reason: {{ cot }}
    {% endif -%}
    Label: {{ label }}
    </EXAMPLE>""",
    line_m_before=3,
    line_n_after=3
)
DEFAULT_DETECTION_CLASS = 'no'

In [14]:
class HuggingFaceModel(Model):
    def __init__(self, task_type: str, model_uri: str, known_labels: set[str], unmatched_label: str):
        super().__init__(task_type, model_uri, known_labels, unmatched_label)
        self.tokenizer = AutoTokenizer.from_pretrained(model_uri)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_uri, device_map="auto")
        self.train_dataset = None
        self.train_indexes = None

    def fit(self, dataset: Dataset):
        self.train_dataset = dataset

    def predict(self, dataset: Dataset, prompt_template: PromptTemplate, train_strategy: TrainStrategy, n_shot_size: int,
                 verbose: bool = False):
        super().predict_start(dataset)
        label_predictions = []
        for index in range(dataset.num_rows):
            train_indexes = pick_n_shot(self.train_dataset, dataset, index, n_shot_size, train_strategy)
            prompt = self.create_prompt(prompt_template, self.train_dataset, train_indexes, dataset, index)
            tokens = self.tokenizer(prompt, return_tensors="pt")
            tokens['input_ids'] = tokens.input_ids.to(self.model.device)
            input_ids = tokens.input_ids
            # print(len(input_ids[0]))
            # detokenized_text = self.tokenizer.decode(input_ids[0], skip_special_tokens=True)
            # print(detokenized_text)
            output = self.model.generate(input_ids)
            # print(self.tokenizer.decode(outputs[0], skip_special_tokens=False))
            label_predictions.append(self.format_label(self.tokenizer.decode(output[0], skip_special_tokens=True), verbose))
        return super().predict_end(dataset, label_predictions)


In [14]:
# Detect with google/flan-t5-small N-Shots

In [49]:
flan_t5_small_detection_model = HuggingFaceModel('detect', 'google/flan-t5-small', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
flan_t5_small_detection_model.fit(detect_n_shot_dataset)
print_classification_excluding_outlier_repository(flan_t5_small_detection_model.predict(detect_test_dataset,  DETECTION_TEMPLATE,
                              TrainStrategy.N_SHOT_SIMILAR,
                              4, verbose=False))


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

detect with flan-t5-small


Token indices sequence length is longer than the specified maximum sequence length for this model (635 > 512). Running this sequence through the model will result in indexing errors


Test Result:
              precision    recall  f1-score   support

          no      0.977     0.996     0.986      7592
         yes      0.029     0.006     0.009       177

    accuracy                          0.973      7769
   macro avg      0.503     0.501     0.498      7769
weighted avg      0.956     0.973     0.964      7769

Test Result Excluding repository: 69
              precision    recall  f1-score   support

          no      0.987     0.996     0.991      7495
         yes      0.029     0.010     0.015       103

    accuracy                          0.982      7598
   macro avg      0.508     0.503     0.503      7598
weighted avg      0.974     0.982     0.978      7598



# Detect with google/flan-t5-base N-Shots

In [14]:
flan_t5_base_detection_model = HuggingFaceModel('detect', 'google/flan-t5-base', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
flan_t5_base_detection_model.fit(detect_n_shot_dataset)
print_classification_excluding_outlier_repository(flan_t5_base_detection_model.predict(detect_test_dataset, DETECTION_TEMPLATE,
                              TrainStrategy.N_SHOT_SIMILAR,
                              4, verbose=False))


detect with flan-t5-base


Token indices sequence length is longer than the specified maximum sequence length for this model (614 > 512). Running this sequence through the model will result in indexing errors


Test Result:
              precision    recall  f1-score   support

          no      0.975     0.445     0.612      7592
         yes      0.021     0.514     0.041       177

    accuracy                          0.447      7769
   macro avg      0.498     0.480     0.326      7769
weighted avg      0.953     0.447     0.599      7769

Test Result Excluding repository: 69
              precision    recall  f1-score   support

          no      0.979     0.442     0.609      7495
         yes      0.008     0.311     0.015       103

    accuracy                          0.440      7598
   macro avg      0.493     0.376     0.312      7598
weighted avg      0.966     0.440     0.601      7598



# Detect with google/flan-t5-large N-Shots

In [14]:
flan_t5_large_detection_model = HuggingFaceModel('detect', 'google/flan-t5-large', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
flan_t5_large_detection_model.fit(detect_n_shot_dataset)
print_classification_excluding_outlier_repository(flan_t5_large_detection_model.predict(detect_test_dataset, DETECTION_TEMPLATE, TrainStrategy.N_SHOT_TOP, 4,
                              verbose=False))


detect with flan-t5-large


Token indices sequence length is longer than the specified maximum sequence length for this model (612 > 512). Running this sequence through the model will result in indexing errors


Test Result:
              precision    recall  f1-score   support

          no      0.998     0.843     0.914      7592
         yes      0.122     0.932     0.215       177

    accuracy                          0.845      7769
   macro avg      0.560     0.888     0.564      7769
weighted avg      0.978     0.845     0.898      7769

Test Result Excluding repository: 69
              precision    recall  f1-score   support

          no      0.998     0.844     0.915      7495
         yes      0.073     0.893     0.135       103

    accuracy                          0.844      7598
   macro avg      0.536     0.868     0.525      7598
weighted avg      0.986     0.844     0.904      7598



# Detect with google/flan-t5-xl N-Shots

In [14]:
flan_t5_xl_detection_model = HuggingFaceModel('detect', 'google/flan-t5-xl', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
flan_t5_xl_detection_model.fit(detect_n_shot_dataset)
print_classification_excluding_outlier_repository(flan_t5_xl_detection_model.predict(detect_test_dataset, DETECTION_TEMPLATE, TrainStrategy.N_SHOT_TOP, 4,
                              verbose=False))


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

detect with flan-t5-xl


Token indices sequence length is longer than the specified maximum sequence length for this model (612 > 512). Running this sequence through the model will result in indexing errors


Test Result:
              precision    recall  f1-score   support

          no      0.999     0.965     0.981      7592
         yes      0.383     0.938     0.544       177

    accuracy                          0.964      7769
   macro avg      0.691     0.951     0.763      7769
weighted avg      0.984     0.964     0.971      7769

Test Result Excluding repository: 69
              precision    recall  f1-score   support

          no      0.998     0.965     0.981      7495
         yes      0.258     0.893     0.401       103

    accuracy                          0.964      7598
   macro avg      0.628     0.929     0.691      7598
weighted avg      0.988     0.964     0.973      7598



# Detect with google/flan-t5-xxl N-Shots

In [ ]:
detect_flan_t5_xxl_model = HuggingFaceModel('detect', 'google/flan-t5-xxl', {'yes', 'no'}, DEFAULT_DETECTION_CLASS, DETECTION_TEMPLATE)
detect_flan_t5_xxl_model.fit(detect_n_shot_dataset)
print_classification_excluding_outlier_repository(detect_flan_t5_xxl_model.predict(detect_test_dataset, TrainStrategy.N_SHOT_TOP, 4,
                              verbose=False))


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

detect with flan-t5-xxl


Token indices sequence length is longer than the specified maximum sequence length for this model (612 > 512). Running this sequence through the model will result in indexing errors


In [ ]:
@retry(
    stop=stop_after_attempt(10),  # Stop after 5 retries
    wait=wait_exponential(multiplier=2, min=60, max=2 * 60),
    retry=retry_if_exception_type(google.api_core.exceptions.ResourceExhausted),  # Retry on rate limit errors
)
def predict_with_gemini(model, prompt):
    generation_config = types.GenerationConfig(
        temperature=0.0
    )
    return model.generate_content(contents=prompt, generation_config=generation_config).text.strip().lower()


class GeminiModel(Model):
    def __init__(self, task_type: str, model_uri: str, known_labels: set[str], unmatched_label: str,
                 prompt_template: PromptTemplate, train_strategy: TrainStrategy, n_shot_size: int,
                 verbose: bool = False):
        super().__init__(task_type, model_uri, known_labels, unmatched_label, verbose)
        self.model = genai.GenerativeModel(model_uri)
        self.prompt_template = prompt_template
        self.train_strategy = train_strategy
        self.n_shot_size = n_shot_size
        self.train_dataset = None
        self.train_indexes = None
        genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

    def fit(self, dataset: Dataset):
        self.train_dataset = dataset

    def predict(self, dataset: Dataset):
        super().predict_start(dataset)
        label_predictions = []
        for index in range(dataset.num_rows):
            train_indexes = pick_n_shot(self.train_dataset, dataset, index, self.n_shot_size, self.train_strategy)
            prompt = self.create_prompt(self.prompt_template, self.train_dataset, train_indexes, dataset, index)
            label_predictions.append(self.format_label(predict_with_gemini(self.model, prompt)))
        return super().predict_end(dataset, label_predictions)


# Detect with Gemini 2.0 Flash N-Shots

In [46]:
gemini_2_flash_detection_model = GeminiModel('detect', 'models/gemini-2.0-flash', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
gemini_2_flash_detection_model.fit(detect_n_shot_dataset)
print_classification_excluding_outlier_repository(gemini_2_flash_detection_model.predict(detect_test_dataset, DETECTION_TEMPLATE, TrainStrategy.N_SHOT_TOP,
                           10, verbose=False))

detect with gemini-2.0-flash
Test Result:
              precision    recall  f1-score   support

          no      1.000     0.959     0.979      7592
         yes      0.363     0.994     0.532       177

    accuracy                          0.960      7769
   macro avg      0.681     0.977     0.755      7769
weighted avg      0.985     0.960     0.969      7769

Test Result Excluding repository: 69
              precision    recall  f1-score   support

          no      1.000     0.959     0.979      7495
         yes      0.249     0.990     0.398       103

    accuracy                          0.959      7598
   macro avg      0.624     0.975     0.688      7598
weighted avg      0.990     0.959     0.971      7598



In [13]:
class ChatGpt4Model(Model):
    def __init__(self, task_type: str, model_uri: str, known_labels: set[str], unmatched_label: str):
        super().__init__(task_type, model_uri, known_labels, unmatched_label, verbose)
        self.client = OpenAI(api_key=os.getenv("OPEN_AI_API_KEY"))
        self.train_dataset = None
        self.train_indexes = None
        genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

    def fit(self, dataset: Dataset):
        self.train_dataset = dataset

    def predict(self, dataset: Dataset, prompt_template: PromptTemplate, train_strategy: TrainStrategy, n_shot_size: int, verbose: bool = False):
        super().predict_start(dataset)
        label_predictions = []
        for index in range(dataset.num_rows):
            train_indexes = pick_n_shot(self.train_dataset, dataset, index, n_shot_size, train_strategy)
            prompt = self.create_prompt(prompt_template, self.train_dataset, train_indexes, dataset, index)
            completion = self.client.chat.completions.create(
                model=self.model_uri,
                store=True,
                messages=[
                    {"role": "user", "content": prompt}
                ])
            label_pred = completion.choices[0].message.content.strip().split()[-1].lower()
            label_predictions.append(self.format_label(label_pred))
        return super().predict_end(dataset, label_predictions)


# Detect with gpt-4o-mini N-Shots

In [102]:
chat_gpt4o_mini_detection_model = ChatGpt4Model('detect', 'gpt-4o-mini', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
chat_gpt4o_mini_detection_model.fit(detect_n_shot_dataset)
print_classification_excluding_outlier_repository(chat_gpt4o_mini_detection_model.predict(detect_test_dataset, DETECTION_TEMPLATE,
                                      TrainStrategy.N_SHOT_TOP, 4, verbose=False))


detect with gpt-4o-mini
              precision    recall  f1-score   support

          no      1.000     0.600     0.750        20
         yes      0.000     0.000     0.000         0

    accuracy                          0.600        20
   macro avg      0.500     0.300     0.375        20
weighted avg      1.000     0.600     0.750        20

Merging Mismatch


'./cache/March 23, 2025, 04:07:07$detect_gpt-4o-mini.csv'

# Detect with gpt-4o N-Shots

In [14]:
chat_gpt4o_detection_model = ChatGpt4Model('detect', 'gpt-4o', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
chat_gpt4o_detection_model.fit(detect_n_shot_dataset)
print_classification_excluding_outlier_repository(chat_gpt4o_detection_model.predict(detect_test_dataset, DETECTION_TEMPLATE,
                                 TrainStrategy.N_SHOT_TOP, 4, verbose=False))


detect with gpt-4o
Unknown Labels:
['classification?', 'analysis?', "'no'?", '```', '</example>', '```', '```', 'code.', 'request.', 'evaluate.', '```', 'debt.', 'classification.', '</example>', 'classification.', 'concerns.', 'classification.', 'you?', 'debt.', '</example>', '</example>', 'request.', 'no.', '```', '```', '</example>', 'itself.', '```', '```', '</example>', 'comment.', '```', 'comments.', 'classified?', 'evaluation.', 'accordingly.', 'request.', 'analysis.', 'request.', '```', '</example>', '```', '```', 'classification.', 'else.', 'analysis.', 'not.', 'confirmed.', "'no'.", 'evaluation.', '```', 'classified?', 'needed?', 'that.', '```no```', '```', 'classification.', 'request.', "'no'.", '</example>', 'analysis.', 'request.', 'evaluated.', 'them.', 'that.', '```no```', 'debt?', '```', '</example>', 'classification.', 'that.', 'that.', 'comments.', 'debt.', 'debt.', 'classification.', '```', 'request.', 'scenario.', '```', 'label.', '```', 'classification.', 'debt.', '

In [64]:
class SentenceEmbeddedLogisticsRegressionModel(Model):
    def __init__(self, task_type: str, model_uri: str, known_labels: set[str], unmatched_label: str):
        super().__init__(task_type, model_uri, known_labels, unmatched_label, verbose)
        self.transformer = SentenceTransformer(model_uri)
        self.model = LogisticRegression()

    def fit(self, dataset: Dataset):
        self.model.fit(self.transformer.encode(dataset['text']), dataset['label'])

    def predict(self, dataset: Dataset, verbose: bool = False):
        super().predict_start(dataset)
        label_predictions = self.model.predict(self.transformer.encode(dataset['text']))
        for index in range(dataset.num_rows):
            label_predictions[index] = self.format_label(label_predictions[index])
        return super().predict_end(dataset, label_predictions)


# Detect with `all-MiniLM-L6-v2` Embedding and Logistic Regression

In [42]:
sentence_embedded_model = SentenceEmbeddedLogisticsRegressionModel('detect', 'all-MiniLM-L6-v2', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
sentence_embedded_model.fit(detect_train_balanced_dataset)
print_classification_excluding_outlier_repository(sentence_embedded_model.predict(detect_test_dataset))

detect with all-MiniLM-L6-v2
              precision    recall  f1-score   support

          no      0.997     0.932     0.963      7592
         yes      0.229     0.864     0.363       177

    accuracy                          0.931      7769
   macro avg      0.613     0.898     0.663      7769
weighted avg      0.979     0.931     0.950      7769

Test Result Excluding repository: 69
              precision    recall  f1-score   support

          no      0.997     0.932     0.963      7495
         yes      0.134     0.767     0.228       103

    accuracy                          0.929      7598
   macro avg      0.565     0.849     0.595      7598
weighted avg      0.985     0.929     0.953      7598



In [16]:
import jpype
import jpype.imports
from jpype.types import *
from dotenv import load_dotenv
import os

load_dotenv()


class TextMiningBasedSatdDetectorModel(Model):
    def __init__(self, task_type: str, model_uri: str, known_labels: set[str], unmatched_label: str):
        super().__init__(task_type, model_uri, known_labels, unmatched_label)

    def fit(self, dataset: Dataset):
        pass

    def predict(self, dataset: Dataset, verbose: bool = False):
        super().predict_start(dataset)
        label_predictions = []

        if not jpype.isJVMStarted():
            jar_path = os.getenv('SATD_DETECTOR_JAR')
            dependency_path = os.getenv('SATD_DETECTOR_DEPENDENCY')
            jvm_args = ["-Xss512m"]
            jpype.startJVM(jpype.getDefaultJVMPath(), classpath=[jar_path, dependency_path], )
        from satd_detector.core.utils import SATDDetector
        detector1 = SATDDetector()
        for index in range(dataset.num_rows):
            label_pred = 'yes' if detector1.isSATD(dataset['text'][index]) else 'no'
            label_predictions.append(self.format_label(label_pred))

        return super().predict_end(dataset, label_predictions)



# Detect with SATD Text Mining Based SATD Detector

In [ ]:
text_mining_based_model = TextMiningBasedSatdDetectorModel('detect', 'text-minig-based-satd-detector', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
text_mining_based_model.fit(detect_train_balanced_dataset)
print_classification_excluding_outlier_repository(text_mining_based_model.predict(detect_test_dataset))

Baseline Model

In [12]:
import shutil
BASE_MAT_INPUT_DIRECTORY = '../cache/baseline/input'
BASE_MAT_OUTPUT_DIRECTORY = '../cache/baseline/output'
os.makedirs(os.path.join(BASE_MAT_INPUT_DIRECTORY, 'origin'), exist_ok=True)
shutil.copytree('../config/baseline', BASE_MAT_INPUT_DIRECTORY, dirs_exist_ok=True)
for kv in [{'train': detect_train_df}, {'test': detect_test_df}, {'merged': pd.concat([detect_train_df.assign(project='train'), detect_test_df.assign(project='test')])}]:
    for df_name, df in kv.items():
        if df_name == 'merged':
            df['text'].str.replace('\n', '\t').to_csv(f'{BASE_MAT_INPUT_DIRECTORY}/origin/comments', index=False, header=False)
            df['label'].str.lower().map({'yes': 'SATD', 'no': 'WITHOUT_CLASSIFICATION'}).to_csv(f'{BASE_MAT_INPUT_DIRECTORY}/origin/labels', index=False, header=False)
            df['project'].to_csv(f'{BASE_MAT_INPUT_DIRECTORY}/origin/projects', index=False, header=False)
        else:
            df['text'].str.replace('\n', '\t').to_csv(f'{BASE_MAT_INPUT_DIRECTORY}/origin/data--{df_name}.txt', index=False, header=False)
            df['label'].str.lower().map({'yes': 'positive', 'no': 'negative'}).to_csv(f'{BASE_MAT_INPUT_DIRECTORY}/origin/label--{df_name}.txt', index=False, header=False)


In [17]:
import jpype
import jpype.imports
from jpype.types import JArray, JString
import os
from dotenv import load_dotenv

load_dotenv()


class BaselineModel(Model):
    def __init__(self, task_type: str, model_uri: str, known_labels: set[str], unmatched_label: str):
        super().__init__(task_type, model_uri, known_labels, unmatched_label)

    def fit(self, dataset: Dataset):
        pass

    def predict(self, dataset: Dataset, verbose: bool = False):
        super().predict_start(dataset)
        print(jpype.isJVMStarted())
        if not jpype.isJVMStarted():
            jar_path = os.path.join(os.getenv('JAR_DETECTOR'), 'MAT.jar')
            # Start JVM
            jpype.startJVM(
            jpype.getDefaultJVMPath(),
            "-ea",  # enable assertions
            f"-Djava.class.path={jar_path}",
            "--add-opens=java.base/java.lang=ALL-UNNAMED")

            # Import Java classes
        from main import Settings
        from main import Main

        Settings.projectNames = JArray(JString)(["train", "test"])

        args = [
            "-p", os.path.join(BASE_MAT_INPUT_DIRECTORY, ''),
            "-o", os.path.join(BASE_MAT_OUTPUT_DIRECTORY, ''),
            "-m", self.model_uri,
            "-s", "MTO"
        ]
        Main.main(JArray(JString)(args))
        #jpype.shutdownJVM()

        predicted_label_df = pd.read_csv(os.path.join(BASE_MAT_OUTPUT_DIRECTORY, f'MTO_{self.model_uri}/result--test.txt'), header=None, names=['label'])
        assert len(predicted_label_df) == len(dataset)
        label_predictions = predicted_label_df['label'].map({0: 'no', 1: 'yes'}).tolist()
        return super().predict_end(dataset, label_predictions)

In [20]:
pattern_model = BaselineModel('detect', 'Pattern', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
pattern_model.fit(detect_test_dataset)
print_classification_excluding_outlier_repository(pattern_model.predict(detect_test_dataset))

detect with Pattern
True
Running model Pattern in MTO
Preparing data for Pattern
Pattern prediction finished!
Method: Pattern
TP, FN, FP, TN, P    , R    , F1   , ER   , RI
14, 469, 8, 37903, 0.636, 0.029, 0.055, 0.980, 49.585
2, 107, 1, 9488, 0.667, 0.018, 0.036, 0.983, 57.703

../cache/August 27, 2025, 17:32:07$detect_Pattern.csv
Test Result:
              precision    recall  f1-score   support

          no      0.989     1.000     0.994      9490
         yes      0.667     0.018     0.036       109

    accuracy                          0.989      9599
   macro avg      0.828     0.509     0.515      9599
weighted avg      0.985     0.989     0.983      9599

Test Result Excluding repository: 69
              precision    recall  f1-score   support

          no      0.989     1.000     0.994      9490
         yes      0.667     0.018     0.036       109

    accuracy                          0.989      9599
   macro avg      0.828     0.509     0.515      9599
weighted avg     

In [21]:
tm_model = BaselineModel('detect', 'TM', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
tm_model.fit(detect_test_dataset)
print_classification_excluding_outlier_repository(tm_model.predict(detect_test_dataset))

detect with TM
True
Running model TM in MTO
Preparing data for Pattern
../cache/baseline/input/tm/data--train.arff
../cache/baseline/input/tm/data--test.arff
Target: train, ../cache/baseline/input/tm/data--train.arff
Target: test, ../cache/baseline/input/tm/data--test.arff
Method: TM
TP, FN, FP, TN, P    , R    , F1   , ER   , RI
393, 90, 2753, 35158, 0.125, 0.814, 0.217, 0.899, 8.930
102, 7, 1010, 8479, 0.092, 0.936, 0.167, 0.876, 7.077

../cache/August 27, 2025, 17:35:55$detect_TM.csv
Test Result:
              precision    recall  f1-score   support

          no      0.999     0.894     0.943      9490
         yes      0.092     0.936     0.167       109

    accuracy                          0.894      9599
   macro avg      0.545     0.915     0.555      9599
weighted avg      0.989     0.894     0.935      9599

Test Result Excluding repository: 69
              precision    recall  f1-score   support

          no      0.999     0.894     0.943      9490
         yes      0.09

In [22]:
nlp_model = BaselineModel('detect', 'NLP', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
nlp_model.fit(detect_test_dataset)
print_classification_excluding_outlier_repository(nlp_model.predict(detect_test_dataset))

detect with NLP
True
Running model NLP in MTO


Setting ColumnDataClassifier properties
sigma = 3
useQN = true
intern = true
1.maxNGramLeng = 4
displayedColumn = 1
trainFile = ./examples/cheeseDisease.train
QNsize = 15
printClassifierParam = 200
1.useNGrams = true
useClassFeature = true
1.usePrefixSuffixNGrams = true
testFile = ./examples/cheeseDisease.test
1.binnedLengths = 10,20,30
goldAnswerColumn = 0
1.minNGramLeng = 1
tolerance = 1e-4
Reading dataset from ../cache/baseline/input/nlp/train--train.txt ... done [3.2s, 9599 items].
numDatums: 9599
numDatumsPerLabel: {WITHOUT_CLASSIFICATION=9490.0, SATD=109.0}
numLabels: 2 [WITHOUT_CLASSIFICATION, SATD]
numFeatures (Phi(X) types): 31313 [CLASS, 1-#-tri, 1-#- unr, 1-#- ma, 1-#-ap, ...]


WITHOUT_CLASSIFICATION	 oauth see   ==>  WITHOUT_CLASSIFICATION (4.8977)
WITHOUT_CLASSIFICATION	 query that should return nothing   ==>  WITHOUT_CLASSIFICATION (6.7639)
WITHOUT_CLASSIFICATION	 formatteroff   ==>  WITHOUT_CLASSIFICATION (5.5527)
WITHOUT_CLASSIFICATION	 formatteron   ==>  WITHOUT_CLASSIFICATION (5.6588)
WITHOUT_CLASSIFICATION	 loads the credentials that will expired soon   ==>  WITHOUT_CLASSIFICATION (7.2434)
WITHOUT_CLASSIFICATION	 these permissions all imply describe   ==>  WITHOUT_CLASSIFICATION (5.4266)
WITHOUT_CLASSIFICATION	 job entities created per job timer and executable job   ==>  WITHOUT_CLASSIFICATION (5.6097)
WITHOUT_CLASSIFICATION	 then   ==>  WITHOUT_CLASSIFICATION (5.2903)
WITHOUT_CLASSIFICATION	 formatteron   ==>  WITHOUT_CLASSIFICATION (5.6588)
WITHOUT_CLASSIFICATION	 connector will return supported from   ==>  WITHOUT_CLASSIFICATION (3.5952)
WITHOUT_CLASSIFICATION	 with dst   ==>  WITHOUT_CLASSIFICATION (4.7119)
WITHOUT_CLASSIFICATION	 the event handle

Built this classifier: LinearClassifier with 31313 features, 2 classes, and 62626 parameters.


WITHOUT_CLASSIFICATION	 shutdown and subchannel state change can happen simultaneously shutdown runs first any further balancing state update should ignored   ==>  WITHOUT_CLASSIFICATION (4.5601)
WITHOUT_CLASSIFICATION	 turn client request validation   ==>  WITHOUT_CLASSIFICATION (5.7795)
WITHOUT_CLASSIFICATION	 starting two instances   ==>  WITHOUT_CLASSIFICATION (5.6416)
WITHOUT_CLASSIFICATION	 create table then get the single region for our new table   ==>  WITHOUT_CLASSIFICATION (7.6449)
WITHOUT_CLASSIFICATION	 copyright camunda services gmbh andor licensed camunda services gmbh under one more contributor license agreements see the notice file distributed with this work for additional information regarding copyright ownership licensed under the camunda license you may not use this file except compliance with the camunda license   ==>  WITHOUT_CLASSIFICATION (5.4780)
WITHOUT_CLASSIFICATION	 then waiting should finish immediately   ==>  WITHOUT_CLASSIFICATION (5.9721)
WITHOUT_CLASSIF

Setting ColumnDataClassifier properties
sigma = 3
useQN = true
intern = true
1.maxNGramLeng = 4
displayedColumn = 1
trainFile = ./examples/cheeseDisease.train
QNsize = 15
printClassifierParam = 200
1.useNGrams = true
useClassFeature = true
1.usePrefixSuffixNGrams = true
testFile = ./examples/cheeseDisease.test
1.binnedLengths = 10,20,30
goldAnswerColumn = 0
1.minNGramLeng = 1
tolerance = 1e-4
Reading dataset from ../cache/baseline/input/nlp/train--test.txt ... done [6.3s, 38395 items].
numDatums: 38395
numDatumsPerLabel: {WITHOUT_CLASSIFICATION=37912.0, SATD=483.0}
numLabels: 2 [WITHOUT_CLASSIFICATION, SATD]
numFeatures (Phi(X) types): 51512 [CLASS, 1-#-th , 1-#B- , 1-#-o, 1-#E- , ...]
Built this classifier: LinearClassifier with 51512 features, 2 classes, and 103024 parameters.


WITHOUT_CLASSIFICATION	 masking applied the unrestricted table   ==>  WITHOUT_CLASSIFICATION (7.6309)
WITHOUT_CLASSIFICATION	 inheritdoc   ==>  WITHOUT_CLASSIFICATION (5.7251)
WITHOUT_CLASSIFICATION	 write from server client with oshut   ==>  WITHOUT_CLASSIFICATION (6.4661)
WITHOUT_CLASSIFICATION	 copyright hazelcast inc all rights reserved licensed under the apache license version the license you may not use this file except compliance with the license you may obtain copy the license unless required applicable law agreed writing software distributed under the license distributed basis without warranties conditions any kind either express implied see the license for the specific language governing permissions and limitations under the license   ==>  WITHOUT_CLASSIFICATION (9.1970)
WITHOUT_CLASSIFICATION	 licensed the apache software foundation asf under one more contributor license agreements see the notice file distributed with this work for additional information regarding copyright 

In [23]:
mat_model = BaselineModel('detect', 'MAT', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
mat_model.fit(detect_test_dataset)
print_classification_excluding_outlier_repository(mat_model.predict(detect_test_dataset))

detect with MAT
True
Running model MAT in MTO
Preparing data for Pattern
../cache/baseline/input/tm/data--train.arff
../cache/baseline/input/tm/data--test.arff
MAT prediction finished!
Method: MAT
TP, FN, FP, TN, P    , R    , F1   , ER   , RI
323, 160, 19, 37892, 0.944, 0.669, 0.783, 0.987, 74.075
76, 33, 1, 9488, 0.987, 0.697, 0.817, 0.988, 85.911

../cache/August 27, 2025, 17:37:34$detect_MAT.csv
Test Result:
              precision    recall  f1-score   support

          no      0.997     1.000     0.998      9490
         yes      0.987     0.697     0.817       109

    accuracy                          0.996      9599
   macro avg      0.992     0.849     0.908      9599
weighted avg      0.996     0.996     0.996      9599

Test Result Excluding repository: 69
              precision    recall  f1-score   support

          no      0.997     1.000     0.998      9490
         yes      0.987     0.697     0.817       109

    accuracy                          0.996      9599
  

In [ ]:
import ast
from util import sha1


@retry(
    stop=stop_after_attempt(10),  # Stop after 5 retries
    wait=wait_exponential(multiplier=2, min=60, max=2 * 60),
    retry=retry_if_exception_type(google.api_core.exceptions.ResourceExhausted))
def create_embedding(uri, text):
    return genai.embed_content(
        model=uri,
        content=text,
        task_type="classification")


class GeminiSentenceTransformer:
    def __init__(self, uri: str, use_cache=False):
        self.uri = uri
        self.use_cache = use_cache
        self.file = f'./cache/{uri.split("/")[-1]}.csv'
        if not os.path.exists(self.file):
            with open(self.file, "w") as file:
                file.write("text,hash,embedding")
                file.flush()
        self.cache_df = pd.read_csv(self.file)
        self.encoding_map = {r['hash']: np.array(ast.literal_eval(r['embedding']), dtype=np.float32) for row_index, r in
                             self.cache_df.iterrows()}
        genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

    def encode(self, features):
        encoded_features = []
        rows = []
        try:
            for text in features:
                if self.use_cache:
                    hash = sha1(text)
                    # embedding_df = self.cache_df[self.cache_df['hash'] == hash]['embedding']
                    if hash not in self.encoding_map:
                        # if embedding_df.empty:
                        response = create_embedding(self.uri, text)
                        embedding_value = response["embedding"]
                        rows.append([text, hash, embedding_value])
                        # self.cache_df.loc[len(self.cache_df)] = [text, hash, embedding_value]
                        # self.cache_df.to_csv(self.file, index=False)
                        # self.cache_df = pd.read_csv(self.file)
                        self.encoding_map[hash] = np.array(embedding_value)
                    encoded_features.append(self.encoding_map[hash])

                    # else:
                    # encoded_features.append(np.array(ast.literal_eval(embedding_df.iloc[0]), dtype=np.float32))
                else:
                    raise Exception('Not Implemented Yet')
        except Exception as e:
            raise e
        finally:
            self.checkpoint(rows)
        return encoded_features

    def checkpoint(self, rows):
        if len(rows) > 0:
            new_df = pd.DataFrame(rows, columns=self.cache_df.columns)
            pd.concat([self.cache_df, new_df]).to_csv(self.file, index=False)
            self.cache_df = pd.read_csv(self.file)


class GeminiEmbeddedLogisticsRegressionModel(Model):
    def __init__(self, task_type: str, model_uri: str, known_labels: set[str], unmatched_label: str):
        super().__init__(task_type, model_uri, known_labels, unmatched_label)
        self.transformer = GeminiSentenceTransformer(model_uri, True)
        self.model = LogisticRegression()

    def fit(self, dataset: Dataset):
        self.model.fit(self.transformer.encode(dataset['text']), dataset['label'])

    def predict(self, dataset: Dataset, verbose: bool = False):
        super().predict_start(dataset)
        label_predictions = self.model.predict(self.transformer.encode(dataset['text']))
        for index in range(dataset.num_rows):
            label_predictions[index] = self.format_label(label_predictions[index])
        return super().predict_end(dataset, label_predictions)


# Detect with `text-embedding-004` Embedding and Logistic Regression

In [20]:
gemini_embedded_004_model = GeminiEmbeddedLogisticsRegressionModel('detect', 'models/text-embedding-004', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
gemini_embedded_004_model.fit(detect_train_balanced_dataset)
print_classification_excluding_outlier_repository(gemini_embedded_004_model.predict(Dataset.from_pandas(detect_test_df[:])))

detect with text-embedding-004
Test Result:
              precision    recall  f1-score   support

          no      0.997     0.952     0.974      7592
         yes      0.295     0.870     0.441       177

    accuracy                          0.950      7769
   macro avg      0.646     0.911     0.707      7769
weighted avg      0.981     0.950     0.962      7769

Test Result Excluding repository: 69
              precision    recall  f1-score   support

          no      0.997     0.953     0.975      7495
         yes      0.186     0.777     0.301       103

    accuracy                          0.951      7598
   macro avg      0.592     0.865     0.638      7598
weighted avg      0.986     0.951     0.965      7598



# Detect with `models/gemini-embedding-exp-03-07` Embedding and Logistic Regression

In [91]:
gemini_embedded_exp_model = GeminiEmbeddedLogisticsRegressionModel('detect', 'models/gemini-embedding-exp-03-07',
                                                                   {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
gemini_embedded_exp_model.fit(detect_train_balanced_dataset)
print_classification_excluding_outlier_repository(gemini_embedded_exp_model.predict(Dataset.from_pandas(detect_test_df[:])))

detect with gemini-embedding-exp-03-07
              precision    recall  f1-score   support

          no      0.996     0.988     0.992      1949
         yes      0.657     0.863     0.746        51

    accuracy                          0.985      2000
   macro avg      0.827     0.925     0.869      2000
weighted avg      0.988     0.985     0.986      2000

Merging Mismatch


'./cache/March 23, 2025, 03:38:26$detect_gemini-embedding-exp-03-07.csv'

# Classification Label Set

In [19]:
classification_label_set = set(classify_train_dataset['label']) | set(classify_test_dataset['label'])
classification_label_set

{'build',
 'code',
 'defect',
 'dependency',
 'design',
 'documentation',
 'how-to',
 'impractical-case',
 'multi',
 'refactor',
 'requirement',
 'skip-test',
 'subset-test',
 'superficial-test',
 'temporary-fix'}

# Classify with `all-MiniLM-L6-v2` Embedding and Logistic Regression

In [65]:
sentence_embedded_classification_model = SentenceEmbeddedLogisticsRegressionModel('classify', 'all-MiniLM-L6-v2', classification_label_set, DEFAULT_CLASSIFICATION_CLASS)
sentence_embedded_classification_model.fit(classify_train_dataset)
print_classification_excluding_outlier_repository(sentence_embedded_classification_model.predict(classify_test_dataset))

classify with all-MiniLM-L6-v2
./cache/March 27, 2025, 17:07:22$classify_all-MiniLM-L6-v2.csv
Test Result:
                  precision    recall  f1-score   support

           build      0.000     0.000     0.000         1
            code      0.000     0.000     0.000         7
          defect      1.000     0.050     0.095        20
      dependency      0.000     0.000     0.000         6
          design      0.000     0.000     0.000         2
   documentation      0.000     0.000     0.000         6
          how-to      0.286     0.091     0.138        22
impractical-case      0.000     0.000     0.000         7
           multi      0.000     0.000     0.000         1
        refactor      0.000     0.000     0.000         1
     requirement      0.672     0.978     0.796       182
       skip-test      0.000     0.000     0.000         7
superficial-test      0.000     0.000     0.000         8
   temporary-fix      0.250     0.136     0.176        22

        accuracy     

# Classify with `models/text-embedding-004` Embedding and Logistic Regression

In [68]:
gemini_embedded_004_classification_model = GeminiEmbeddedLogisticsRegressionModel('classify', 'models/text-embedding-004',  classification_label_set, DEFAULT_CLASSIFICATION_CLASS)
gemini_embedded_004_classification_model.fit(classify_train_dataset)
print_classification_excluding_outlier_repository(gemini_embedded_004_classification_model.predict(classify_test_dataset))

classify with text-embedding-004
./cache/March 27, 2025, 17:10:18$classify_text-embedding-004.csv
Test Result:
                  precision    recall  f1-score   support

           build      0.000     0.000     0.000         1
            code      0.000     0.000     0.000         7
          defect      1.000     0.050     0.095        20
      dependency      0.000     0.000     0.000         6
          design      0.000     0.000     0.000         2
   documentation      0.000     0.000     0.000         6
          how-to      0.000     0.000     0.000        22
impractical-case      0.000     0.000     0.000         7
           multi      0.000     0.000     0.000         1
        refactor      0.000     0.000     0.000         1
     requirement      0.631     0.995     0.772       182
       skip-test      0.000     0.000     0.000         7
superficial-test      0.000     0.000     0.000         8
   temporary-fix      0.500     0.045     0.083        22

        accuracy 

In [ ]:
CLASSIFICATION_TEMPLATE = PromptTemplate(
    name="Manually Crafted Classification",
    definition="""
You are a experienced software engineer reviewing code comments to classify test code comments that indicate Self-Admitted Technical Debt (SATD).
A comment is considered SATD if a developer explicitly acknowledges that the code requires future work. Your task is to classify SATD comments into one of the 15 predefined categories below.

Categories of Self-Admitted Technical Debt (SATD):

Build : Issues related to the build process, such as poorly defined buid environment.

Code: Refers to poor naming conventions, unhandled or ignored exceptions, and the use of inefficient or slow algorithms.

Defect: Refers to comments that exclusively identify unresolved known defects, test case failures, or unexpected results, without referencing other types of technical debt.

Design: Refers to practices that violate the principles of good object-oriented design, such as high coupling and low cohesion.

Dependency: Refers to situations where test code relies on external dependencies, such as libraries, APIs, or services, that hinder the execution of tests due to insufficient support or incomplete functionality.

Documentation: Documentation Debt refers to the lack, incompleteness, or outdated state of code documentation.

How-To: This debt arises when there is uncertainty or a lack of clarity about how to implement, test, or resolve an issue in the code. This type of debt is characterized by unanswered questions, ambiguous comments, or speculative reasoning about expected behavior.

Impractical-Case: Refers to unexpected or problematic situations that are not anticipated under normal circumstances, yet are explicitly mentioned in comments.

Refactor: Arises when code requires restructuring, including code duplication, refactoring, cleanup, and the removal of unnecessary code. This type of debt often exists independently and does not rely on the resolution of other technical debt.

Multi: Refers to a form of technical debt that combines multiple types of debt.

Requirement: Occurs when test cases are incomplete or lack proper implementation. This debt often indicates that something needs to be done without providing specific details about the task.

Skip-Test: This debt arises when certain tests are skipped, disabled.

Subset-Test: A subset test is a special form of Skip-Test where only a small, representative portion of the full input space is used to verify the functionality, often for the sake of time efficiency or resource constraints.

Superficial-Test: This debt indicates partial or inadequate coverage of testing.

Temporary-Fix: A temporary solution implemented to address an issue, intended to be replaced with a permanent fix. It is often implicitly or explicitly mentioned as temporary due to dependencies on other issue.

    """,
    instruction="Classify the following test code comment into one of the above 15 categories. Output only the label corresponding to the most suitable category.",
    n_shot_template="""
Comment: {{ text }}
{% if cot -%}
Reason: {{ cot }}
{% endif -%}
Label: {{ label }}""",
    line_m_before=3,
    line_n_after=10
)
DEFAULT_CLASSIFICATION_CLASS = 'requirement'

# Classify with google/flan-t5-large n-Shots

In [18]:
classify_flan_t5_large_model = HuggingFaceModel('classify', 'google/flan-t5-large', classification_label_set, DEFAULT_CLASSIFICATION_CLASS)
classify_flan_t5_large_model.fit(classify_n_shot_dataset)
print_classification_excluding_outlier_repository(classify_flan_t5_large_model.predict(classify_test_dataset, CLASSIFICATION_TEMPLATE, TrainStrategy.N_SHOT_TOP, 0,
                              verbose=False))


Token indices sequence length is longer than the specified maximum sequence length for this model (688 > 512). Running this sequence through the model will result in indexing errors


classify with flan-t5-large
Unknown Labels:
['Deficit', 'Deficit', 'Deficit', 'Deficit']
Test Result:
                  precision    recall  f1-score   support

           build      1.000     1.000     1.000         1
            code      0.000     0.000     0.000         7
          defect      0.257     0.450     0.327        20
      dependency      0.143     0.167     0.154         6
          design      0.000     0.000     0.000         2
   documentation      1.000     0.333     0.500         6
          how-to      0.800     0.182     0.296        22
impractical-case      0.400     0.286     0.333         7
           multi      0.000     0.000     0.000         1
        refactor      0.000     0.000     0.000         1
     requirement      0.785     0.841     0.812       182
       skip-test      0.000     0.000     0.000         7
     subset-test      0.000     0.000     0.000         0
superficial-test      0.000     0.000     0.000         8
   temporary-fix      0.000

# Classify with google/flan-t5-xl n-Shots

In [15]:
classify_flan_t5_xl_model = HuggingFaceModel('classify', 'google/flan-t5-xl', classification_label_set, DEFAULT_CLASSIFICATION_CLASS)
classify_flan_t5_xl_model.fit(classify_n_shot_dataset)
print_classification_excluding_outlier_repository(classify_flan_t5_xl_model.predict(classify_test_dataset, CLASSIFICATION_TEMPLATE, TrainStrategy.N_SHOT_TOP, 0,
                              verbose=False))

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (617 > 512). Running this sequence through the model will result in indexing errors


classify with flan-t5-xl
Test Result:
                  precision    recall  f1-score   support

           build      0.500     0.667     0.571         3
            code      1.000     0.111     0.200         9
          defect      0.750     0.300     0.429        20
      dependency      0.188     0.333     0.240         9
          design      0.000     0.000     0.000         0
   documentation      1.000     0.250     0.400         4
          how to      0.364     0.400     0.381        20
impractical case      0.000     0.000     0.000         1
           multi      0.043     0.333     0.077         3
        refactor      0.079     0.778     0.143         9
     requirement      0.955     0.626     0.756       171
       skip test      1.000     0.235     0.381        17
     subset test      1.000     0.500     0.667         2
superficial test      0.000     0.000     0.000         4
   temporary fix      1.000     0.100     0.182        20

        accuracy                

# Classify with google/flan-t5-xxl n-Shots

In [ ]:
classify_flan_t5_xxl_model = HuggingFaceModel('classify', 'google/flan-t5-xxl', classification_label_set, DEFAULT_CLASSIFICATION_CLASS)
classify_flan_t5_xxl_model.fit(classify_n_shot_dataset)
print_classification_excluding_outlier_repository(classify_flan_t5_xxl_model.predict(classify_test_dataset, CLASSIFICATION_TEMPLATE, TrainStrategy.N_SHOT_TOP, 0,
                              verbose=False))


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
Token indices sequence length is longer than the specified maximum sequence length for this model (616 > 512). Running this sequence through the model will result in indexing errors


classify with flan-t5-xxl


# Classify with Gemini Flash 2.0 N-Shots

In [19]:
gemini_flash_2_classification_model = GeminiModel('classify', 'models/gemini-2.0-flash', classification_label_set, DEFAULT_CLASSIFICATION_CLASS)
gemini_flash_2_classification_model.fit(classify_n_shot_dataset)
print_classification_excluding_outlier_repository(gemini_flash_2_classification_model.predict(classify_test_dataset, CLASSIFICATION_TEMPLATE, TrainStrategy.N_SHOT_TOP,
                           0, verbose=False))

classify with gemini-2.0-flash
Test Result:
                  precision    recall  f1-score   support

           build      0.500     1.000     0.667         1
            code      0.150     0.429     0.222         7
          defect      0.667     0.600     0.632        20
      dependency      0.333     0.333     0.333         6
          design      0.000     0.000     0.000         2
   documentation      0.750     1.000     0.857         6
          how-to      0.700     0.636     0.667        22
impractical-case      1.000     0.286     0.444         7
           multi      0.000     0.000     0.000         1
        refactor      0.000     0.000     0.000         1
     requirement      0.926     0.896     0.911       182
       skip-test      0.417     0.714     0.526         7
     subset-test      0.000     0.000     0.000         0
superficial-test      0.400     0.500     0.444         8
   temporary-fix      0.500     0.182     0.267        22

        accuracy          

# Classify with gpt-4o-mini N-Shots

In [21]:
gpt_4o_mini_classification_model = ChatGpt4Model('classify', 'gpt-4o-mini', classification_label_set, DEFAULT_CLASSIFICATION_CLASS)
gpt_4o_mini_classification_model.fit(classify_n_shot_dataset)
print_classification_excluding_outlier_repository(gpt_4o_mini_classification_model.predict(classify_test_dataset, CLASSIFICATION_TEMPLATE, TrainStrategy.N_SHOT_TOP,
                           0, verbose=False))

classify with gpt-4o-mini
Unknown Labels:
['**dependency**', '**documentation**', '**how-to**', '**requirement**', '**temporary-fix**', '**how-to**', '**multi**', '**code**', '**documentation**', '**documentation**', '**multi**']
./cache/March 27, 2025, 18:01:56$classify_gpt-4o-mini.csv
Test Result:
                  precision    recall  f1-score   support

           build      0.200     1.000     0.333         1
            code      0.111     0.143     0.125         7
          defect      0.611     0.550     0.579        20
      dependency      0.188     0.500     0.273         6
          design      0.125     1.000     0.222         2
   documentation      0.333     0.833     0.476         6
          how-to      0.600     0.409     0.486        22
impractical-case      0.200     0.429     0.273         7
           multi      0.000     0.000     0.000         1
        refactor      0.000     0.000     0.000         1
     requirement      0.901     0.802     0.849       182
  

# Classify with gpt-4o N-Shots

In [22]:
gpt_4o_classification_model = ChatGpt4Model('classify', 'gpt-4o', classification_label_set, DEFAULT_CLASSIFICATION_CLASS)
gpt_4o_classification_model.fit(classify_n_shot_dataset)
print_classification_excluding_outlier_repository(gpt_4o_classification_model.predict(classify_test_dataset, CLASSIFICATION_TEMPLATE, TrainStrategy.N_SHOT_TOP,
                           0, verbose=False))

classify with gpt-4o
Unknown Labels:
['**temporary-fix**', '**temporary-fix**', '**dependency**', '**dependency**']
./cache/March 27, 2025, 18:07:16$classify_gpt-4o.csv
Test Result:
                  precision    recall  f1-score   support

           build      0.500     1.000     0.667         1
            code      0.300     0.429     0.353         7
          defect      0.636     0.350     0.452        20
      dependency      0.158     0.500     0.240         6
          design      0.167     0.500     0.250         2
   documentation      1.000     1.000     1.000         6
          how-to      0.722     0.591     0.650        22
impractical-case      0.214     0.429     0.286         7
           multi      0.000     0.000     0.000         1
        refactor      0.000     0.000     0.000         1
     requirement      0.905     0.890     0.898       182
       skip-test      0.600     0.857     0.706         7
     subset-test      0.000     0.000     0.000         0
super